In [48]:
import os
import torch
import data_provider
from tqdm import tqdm
import re

In [ ]:
# def filter_duplication_translation(train_dataset, val_dataset, test_dataset, task=None):
#     """
#     input and output in val_dataset and test_dataset should not be in train_dataset
    
#     T2M
#         input: description
#         output: mol
#     M2T
#         input: mol
#         output: description
    
    
#     each data tuple: graph, label, input_mol_string, self.task_subtask_pair, instruction
#     """
    
#     assert task is not None, "task should be specified for translation"
    
#     def extract_description(data):
#         try:
#             return re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", data, re.DOTALL).group()
#         except:
#             print(f"Error in extracting description: {data}")
#             raise ValueError("Error in extracting description")
#     def extract_selfies(data):
#         return re.search(r"(?<=<SELFIES>).*?(?=</SELFIES>)", data).group()
    
    
#     train_label = []
#     train_input = []
    
#     if task == 'T2M':
#         # for instance in train_dataset:
#         for instance in tqdm(train_dataset):
#             train_label.append(extract_selfies(instance[1]))
#             train_input.append(extract_description(instance[4]))
        
#         val_label = list(map(lambda x: extract_selfies(x), val_dataset[:][1]))
#         test_label = list(map(lambda x: extract_selfies(x), test_dataset[:][1]))
        
#         val_input = list(map(lambda x: extract_description(x), val_dataset[:][4]))
#         test_input = list(map(lambda x: extract_description(x), test_dataset[:][4]))
        
#     elif task == 'M2T':
#         for instance in train_dataset:    
#             train_label.append(extract_description(instance[1]))
#             train_input.append(extract_selfies(instance[2]))
        
#         val_label = list(map(lambda x: extract_description(x), val_dataset[:][1]))
#         test_label = list(map(lambda x: extract_description(x), test_dataset[:][1]))
        
#         val_input = list(map(lambda x: extract_selfies(x), val_dataset[:][2]))
#         test_input = list(map(lambda x: extract_selfies(x), test_dataset[:][2]))

#     else:
#         raise NotImplementedError("task should be either T2M or M2T for removing duplication in translation task")

#     # Convert val_input, val_label, test_input, and test_label to sets for fast lookup
#     val_input_set = set(val_input)
#     val_label_set = set(val_label)
#     test_input_set = set(test_input)
#     test_label_set = set(test_label)

#     train_idxs = [
#         idx for idx, (train_in, train_lb) in enumerate(zip(train_input, train_label))
#         if train_in not in val_input_set and train_in not in test_input_set and
#         train_lb not in val_label_set and train_lb not in test_label_set
#     ]
#     """
#     train_idxs_test = [
#         idx for idx, (train_in, train_lb) in enumerate(zip(train_input, train_label))
#         if train_in not in test_input_set and train_lb not in test_label_set
#     ]
#     """

#     print(f"Number of duplicated data: {len(train_dataset) - len(train_idxs)} for {task}")

#     filtered_train_dataset = Subset(train_dataset, train_idxs)
    
#     return filtered_train_dataset

In [16]:


base_path = "/home/chanhui-lee/text-mol/MolCA/data/multi_task_0927/processed/"

ver_tag = 'v2'


train_path = f"mistralai_Mistral-7B-Instruct-v0.3_extended-{ver_tag}_train.pt"
test_path = f"mistralai_Mistral-7B-Instruct-v0.3_extended-{ver_tag}_test.pt"


In [17]:
"""
key:
'x', 'edge_index', 'edge_attr', 'additional_edge_index', 'additional_edge_attr', 'additional_x', 
'input_text', 'target_text', 'prompt_text', 
'task_subtask_pair', 'input_ids'

"""

test_data = torch.load(os.path.join(base_path, test_path))
print("test data loaded")


train_data = torch.load(os.path.join(base_path, train_path))
print("train data loaded")

/tmp/ipykernel_390249/1873209917.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_data = torch.load(os.path.join(base_path, test_path))


test data loaded


/tmp/ipykernel_390249/1873209917.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = torch.load(os.path.join(base_path, train_path))


train data loaded


In [47]:
print(train_data[0].keys())

dict_keys(['x', 'edge_index', 'edge_attr', 'additional_edge_attr', 'additional_edge_index', 'additional_x', 'input_text', 'target_text', 'task_subtask_pair', 'input_ids'])


In [18]:
print(sorted(list(set(train_data[0]['task_subtask_pair']))))
print(sorted(list(set(test_data[0]['task_subtask_pair']))))
print()
print()

['bace/Class', 'bbbp/p_np', 'chebi-20-mol2text/chebi-20-mol2text', 'chebi-20-text2mol/chebi-20-text2mol', 'clintox/CT_TOX', 'clintox/FDA_APPROVED', 'esol/measured log solubility in mols per litre', 'forward_reaction_prediction/forward_reaction_prediction', 'hiv/HIV_active', 'lipo/exp', 'qm9_additional_label/alpha', 'qm9_additional_label/cv', 'qm9_additional_label/g298', 'qm9_additional_label/h298', 'qm9_additional_label/mu', 'qm9_additional_label/r2', 'qm9_additional_label/u298', 'qm9_additional_label/zpve', 'qm9_homo/qm9_homo', 'qm9_homo_lumo_gap/qm9_homo_lumo_gap', 'qm9_lumo/qm9_lumo', 'reagent_prediction/reagent_prediction', 'retrosynthesis/retrosynthesis', 'sider/Hepatobiliary disorders', 'smol-forward_synthesis/smol-forward_synthesis', 'smol-molecule_captioning/smol-molecule_captioning', 'smol-molecule_generation/smol-molecule_generation', 'smol-name_conversion-i2f/smol-name_conversion-i2f', 'smol-name_conversion-i2s/smol-name_conversion-i2s', 'smol-name_conversion-s2f/smol-name_c

In [19]:
set(train_data[0]['task_subtask_pair'])

{'bace/Class',
 'bbbp/p_np',
 'chebi-20-mol2text/chebi-20-mol2text',
 'chebi-20-text2mol/chebi-20-text2mol',
 'clintox/CT_TOX',
 'clintox/FDA_APPROVED',
 'esol/measured log solubility in mols per litre',
 'forward_reaction_prediction/forward_reaction_prediction',
 'hiv/HIV_active',
 'lipo/exp',
 'qm9_additional_label/alpha',
 'qm9_additional_label/cv',
 'qm9_additional_label/g298',
 'qm9_additional_label/h298',
 'qm9_additional_label/mu',
 'qm9_additional_label/r2',
 'qm9_additional_label/u298',
 'qm9_additional_label/zpve',
 'qm9_homo/qm9_homo',
 'qm9_homo_lumo_gap/qm9_homo_lumo_gap',
 'qm9_lumo/qm9_lumo',
 'reagent_prediction/reagent_prediction',
 'retrosynthesis/retrosynthesis',
 'sider/Hepatobiliary disorders',
 'smol-forward_synthesis/smol-forward_synthesis',
 'smol-molecule_captioning/smol-molecule_captioning',
 'smol-molecule_generation/smol-molecule_generation',
 'smol-name_conversion-i2f/smol-name_conversion-i2f',
 'smol-name_conversion-i2s/smol-name_conversion-i2s',
 'smol-na

In [20]:
set(test_data[0]['task_subtask_pair'])

{'bace/Class',
 'bbbp/p_np',
 'chebi-20-mol2text/chebi-20-mol2text',
 'chebi-20-text2mol/chebi-20-text2mol',
 'clintox/CT_TOX',
 'clintox/FDA_APPROVED',
 'esol/measured log solubility in mols per litre',
 'forward_reaction_prediction/forward_reaction_prediction',
 'hiv/HIV_active',
 'lipo/exp',
 'qm9_homo/qm9_homo',
 'qm9_homo_lumo_gap/qm9_homo_lumo_gap',
 'qm9_lumo/qm9_lumo',
 'reagent_prediction/reagent_prediction',
 'retrosynthesis/retrosynthesis',
 'sider/Hepatobiliary disorders',
 'smol-forward_synthesis/smol-forward_synthesis',
 'smol-molecule_captioning/smol-molecule_captioning',
 'smol-molecule_generation/smol-molecule_generation',
 'smol-retrosynthesis/smol-retrosynthesis',
 'tox21/NR-AR',
 'toxcast/ACEA_T47D_80hr_Negative'}

In [22]:
test_data[0].keys()

dict_keys(['x', 'edge_index', 'edge_attr', 'additional_edge_index', 'additional_edge_attr', 'additional_x', 'input_text', 'target_text', 'prompt_text', 'task_subtask_pair', 'input_ids'])

In [50]:
from collections import defaultdict
# def get_filtered_data(data, task_subtask_pair, filter_key=['input_text', 'target_text', 'prompt_text', 'task_subtask_pair']):
def get_filtered_data(data, task_subtask_pair, filter_key=['input_text', 'target_text', 'task_subtask_pair']):
    key_to_search = 'task_subtask_pair'
    value_to_find = task_subtask_pair
    # index = data[key_to_search].index(value_to_find)
    indices = [i for i, x in enumerate(data[key_to_search]) if x == value_to_find]
    
    results = defaultdict(list)
    
    for key in filter_key:
        for idx in indices:
            results[key].append(data[key][idx])
        
    
    return results


key_to_search = 'task_subtask_pair'
value_to_find = 'smol-molecule_generation/smol-molecule_generation'

# index = test_data[0][key_to_search].index(value_to_find)
# indices = [i for i, x in enumerate(test_data[0][key_to_search]) if x == value_to_find]
# print(indices)
# t2m_test_dict_smol = {key: values[index] for key, values in test_data[0].items()}
t2m_test_dict_smol = get_filtered_data(test_data[0], value_to_find)

value_to_find = 'chebi-20-text2mol/chebi-20-text2mol'
# index = test_data[0][key_to_search].index(value_to_find)
# indices = [i for i, x in enumerate(test_data[0][key_to_search]) if x == value_to_find]
# print(indices)
# t2m_test_dict_che = {key: values[index] for key, values in test_data[0].items()}
t2m_test_dict_che = get_filtered_data(test_data[0], value_to_find)



In [38]:
len(t2m_test_dict_smol['input_text']), len(t2m_test_dict_che['input_text'])

(2493, 3297)

In [51]:

t2m_train_dict_smol = get_filtered_data(train_data[0], 'smol-molecule_generation/smol-molecule_generation')
t2m_train_dict_che = get_filtered_data(train_data[0], 'chebi-20-text2mol/chebi-20-text2mol')


In [42]:
len(t2m_train_dict_smol['input_text']), len(t2m_train_dict_che['input_text'])

(34523, 26108)

In [52]:
print(t2m_train_dict_smol['input_text'][0])
print(t2m_train_dict_smol['target_text'][0])
print(t2m_train_dict_che['input_text'][0])
print(t2m_train_dict_che['target_text'][0])

<s> [INST] You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation . </s></s>What is the molecule that can be derived from these structural description?</s> <DESCRIPTION> The molecule is a branched amino pentasaccharide consisting of D-glucose at the reducing end having an alpha-L-fucosyl-(1->3)-[beta-D-galactosyl-(1->4)]-N-acetyl-beta-D-glucosaminyl-(1->3)-beta-D-galactosyl moiety attached at the 4-position . It is an amino pentasaccharide and a glucosamine oligosaccharide . </DESCRIPTION> [/INST]<SELFIES>[C][C][=Branch1][C][=O][N][C@H1][C@H1][Branch2][Ring2][#Branch2][O][C@H1][C@@H1][Branch1][C][O][C@@H1][Branch1][Ring1][C][O][O][C@@H1][Branch2][Ring1][Branch1][O][C@H1][C@H1][Branch1][C][O][C@@H1][Branch1][C][O][C][Branch1][C][O][O][C@@H1][Ring1][=Branch2][C][O][C@@H1][Ring2][Ring1][Branch1][O][O][C@H1][Branch1][Ring1][C][O][

In [29]:
results = defaultdict(list)


In [55]:
desc_pattern = r"<DESCRIPTION>(.*?)</DESCRIPTION>"
selfies_pattern = r"<SELFIES>(.*?)</SELFIES>"
desc = re.findall(desc_pattern, t2m_train_dict_smol['input_text'][0])
sel = re.findall(selfies_pattern, t2m_train_dict_smol['target_text'][0])

print(desc[0].strip())
print(sel[0])

The molecule is a branched amino pentasaccharide consisting of D-glucose at the reducing end having an alpha-L-fucosyl-(1->3)-[beta-D-galactosyl-(1->4)]-N-acetyl-beta-D-glucosaminyl-(1->3)-beta-D-galactosyl moiety attached at the 4-position . It is an amino pentasaccharide and a glucosamine oligosaccharide .
[C][C][=Branch1][C][=O][N][C@H1][C@H1][Branch2][Ring2][#Branch2][O][C@H1][C@@H1][Branch1][C][O][C@@H1][Branch1][Ring1][C][O][O][C@@H1][Branch2][Ring1][Branch1][O][C@H1][C@H1][Branch1][C][O][C@@H1][Branch1][C][O][C][Branch1][C][O][O][C@@H1][Ring1][=Branch2][C][O][C@@H1][Ring2][Ring1][Branch1][O][O][C@H1][Branch1][Ring1][C][O][C@@H1][Branch2][Ring1][Branch1][O][C@@H1][O][C@H1][Branch1][Ring1][C][O][C@H1][Branch1][C][O][C@H1][Branch1][C][O][C@H1][Ring1][#Branch2][O][C@@H1][Ring2][Ring2][O][O][C@@H1][O][C@@H1][Branch1][C][C][C@@H1][Branch1][C][O][C@@H1][Branch1][C][O][C@@H1][Ring1][=Branch2][O]


In [83]:
# def filter_duplication_translation(train_dataset, val_dataset, test_dataset, task=None):
def filter_duplication_translation(train_dataset, test_dataset, task=None):
    """
    input and output in val_dataset and test_dataset should not be in train_dataset
    
    T2M
        input: description
        output: mol
    M2T
        input: mol
        output: description
    
    
    each data tuple: graph, label, input_mol_string, self.task_subtask_pair, instruction
    """
    
    assert task is not None, "task should be specified for translation"
    
    def extract_description(data):
        try:
            return re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", data, re.DOTALL).group()#.strip()
        except:
            print(f"Error in extracting description: {data}")
            raise ValueError("Error in extracting description")
    def extract_selfies(data):
        return re.search(r"(?<=<SELFIES>).*?(?=</SELFIES>)", data).group()
    
    
    train_full_input, train_full_output = train_dataset
    assert len(train_full_input) == len(train_full_output)
    test_full_input, test_full_output = test_dataset    
    assert len(test_full_input) == len(test_full_output)

    if task == 'T2M':
        # for instance in train_dataset:
        # for instance in tqdm(train_dataset):
        #     train_label.append(extract_selfies(instance[1]))
        #     train_input.append(extract_description(instance[4]))
            
        train_label = list(map(lambda x: extract_selfies(x), train_full_output))
        train_input = list(map(lambda x: extract_description(x), train_full_input))
        
        test_label = list(map(lambda x: extract_selfies(x), test_full_output))
        test_input = list(map(lambda x: extract_description(x), test_full_input))
        
    elif task == 'M2T':
        
        train_label = list(map(lambda x: extract_description(x), train_full_output))
        train_input = list(map(lambda x: extract_selfies(x), train_full_input))
        
        test_label = list(map(lambda x: extract_description(x), test_full_output))
        test_input = list(map(lambda x: extract_selfies(x), test_full_input))
        

    else:
        raise NotImplementedError("task should be either T2M or M2T for removing duplication in translation task")

    # Convert val_input, val_label, test_input, and test_label to sets for fast lookup
    # val_input_set = set(val_input)
    # val_label_set = set(val_label)
    test_input_set = set(test_input)
    test_label_set = set(test_label)

    train_input_idxs = [
        idx for idx, (train_in, train_lb) in enumerate(zip(train_input, train_label))
        if train_in not in test_input_set
        # if train_in in test_input_set
        # if train_in not in val_input_set and train_in not in test_input_set and
        # train_lb not in val_label_set and train_lb not in test_label_set
    ]
    train_label_idxs = [
        idx for idx, (train_in, train_lb) in enumerate(zip(train_input, train_label))
        if train_lb not in test_label_set
        # if train_lb in test_label_set
    ]

    # print(f"Number of duplicated data: {len(train_dataset) - len(train_input_idxs)} for {task}")
    
    print("# of prev train set", len(train_dataset[0]))
    print("# of new train input set", len(train_input_idxs), len(train_dataset[0]) - len(train_input_idxs))
    print("# of new train output set", len(train_label_idxs), len(train_dataset[0]) - len(train_label_idxs))
    print("intersction input and lable duplication", len(set(train_input_idxs).intersection(set(train_label_idxs))))

    # filtered_train_dataset = Subset(train_dataset, train_idxs)
    
    # return filtered_train_dataset
    return train_input_idxs, train_label_idxs


# result = filter_duplication_translation(
#     [t2m_train_dict_smol['input_text'], t2m_train_dict_smol['target_text']], 
#     [t2m_test_dict_smol['input_text'], t2m_test_dict_smol['target_text']], 
#     task='T2M')

result = filter_duplication_translation(
    [t2m_train_dict_smol['input_text'], t2m_train_dict_smol['target_text']], 
    [t2m_test_dict_che['input_text'], t2m_test_dict_che['target_text']], 
    task='T2M')

# of prev train set 34523
# of new train input set 33350 1173
# of new train output set 32870 1653
intersction input and lable duplication 32409


In [77]:
temp = re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", t2m_train_dict_smol['input_text'][2], re.DOTALL).group()

for i in range(len(t2m_test_dict_che['input_text'])):
    temp2 = re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", t2m_test_dict_che['input_text'][i], re.DOTALL).group()
    if temp2 == temp:
        print(i)
        break

1863


In [81]:
print(re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", t2m_train_dict_smol['input_text'][2], re.DOTALL).group())
print(re.search(r"(?<=<DESCRIPTION>).*?(?=</DESCRIPTION>)", t2m_test_dict_che['input_text'][1863], re.DOTALL).group())

print(re.search(r"(?<=<SELFIES>).*?(?=</SELFIES>)", t2m_train_dict_smol['target_text'][2], re.DOTALL).group())
print(re.search(r"(?<=<SELFIES>).*?(?=</SELFIES>)", t2m_test_dict_che['target_text'][1863], re.DOTALL).group())

 The molecule is a 1,4-benzodiazepinone that is 1,3-dihydro-2H-1,4-benzodiazepin-2-one substituted by a chloro group at position 7, a methyl group at position 1 and a phenyl group at position 5 . It has a role as a xenobiotic, an environmental contaminant, an anxiolytic drug, an anticonvulsant and a sedative . It is a 1,4-benzodiazepinone and an organochlorine compound . 
 The molecule is a 1,4-benzodiazepinone that is 1,3-dihydro-2H-1,4-benzodiazepin-2-one substituted by a chloro group at position 7, a methyl group at position 1 and a phenyl group at position 5 . It has a role as a xenobiotic, an environmental contaminant, an anxiolytic drug, an anticonvulsant and a sedative . It is a 1,4-benzodiazepinone and an organochlorine compound . 
[C][N][C][=Branch1][C][=O][C][N][=C][Branch1][=Branch2][C][=C][C][=C][C][=C][Ring1][=Branch1][C][=C][C][Branch1][C][Cl][=C][C][=C][Ring1][#Branch1][Ring2][Ring1][Ring1]
[C][N][C][=Branch1][C][=O][C][N][=C][Branch1][=Branch2][C][=C][C][=C][C][=C][Ring